# AI Event Assistant for CAIAS
### Google Colab prototype — academic calendar + personalised recommendations + optional email data

**What this notebook does:**
- Reads the CAIAS Academic Calendar PDF and extracts event rows.
- Lets a student enter their `@caias.in` email and select interests.
- Recommends upcoming events using TF-IDF/cosine similarity plus keyword matching.
- Supports optional event-email import from a CSV or pasted email text.
- Launches a Gradio web interface with a temporary public share link.

**Important prototype limitations:** email-domain checking in this demo is NOT secure authentication. Do not collect passwords. The optional Gmail API connection in a Colab notebook accesses the Google account authorised in that notebook; it does not automatically authenticate every student visiting the Gradio link. Real multi-student Google sign-in/Gmail access requires a registered OAuth web application, consent configuration, and a stable deployed callback URL.


In [ ]:
!pip -q install gradio pymupdf pdfplumber pandas scikit-learn python-dateutil

## 1. Upload the CAIAS academic calendar PDF
Run this cell and choose `CAIAS COE 2026 - 27.pdf` from your device. If you already have it in the Colab session, you can skip upload and set `PDF_PATH` to its path.

In [ ]:
from google.colab import files
uploaded = files.upload()
PDF_PATH = next((name for name in uploaded if name.lower().endswith('.pdf')), None)
if not PDF_PATH:
    raise ValueError('Please upload the CAIAS calendar PDF.')
print('Calendar file:', PDF_PATH)

Saving CAIAS COE 2026 - 27.pdf to CAIAS COE 2026 - 27.pdf
Calendar file: CAIAS COE 2026 - 27.pdf


## 2. Extract event rows from the PDF
The parser handles the calendar's table-like text layout. Always review the extracted table because PDF line wrapping can occasionally split a department or event title.

In [ ]:
import pdfplumber, re, pandas as pd, numpy as np
from datetime import datetime, date

DATE_FORMATS = ['%d-%b-%y', '%d-%b-%Y', '%d-%B-%y', '%d-%B-%Y']

def parse_calendar_date(value):
    value = str(value or '').strip()
    if not value:
        return None
    for fmt in DATE_FORMATS:
        try:
            return pd.Timestamp(datetime.strptime(value, fmt)).normalize()
        except ValueError:
            pass
    return None

def extract_events(pdf_path):
    records = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            for table in page.extract_tables():
                for row in table:
                    if not row or len(row) < 5:
                        continue
                    vals = [re.sub(r'\s+', ' ', str(x or '')).strip() for x in row[:5]]
                    department, start_raw, end_raw, day_name, title = vals
                    start_date = parse_calendar_date(start_raw)
                    if start_date is None:
                        continue  # skip table headers and month headings
                    end_date = parse_calendar_date(end_raw) or start_date
                    title = title.strip() or 'Unspecified event'
                    if title.lower() in {'event title', 'title'}:
                        continue
                    records.append({
                        'department': department or 'Unclassified',
                        'start_date': start_date,
                        'end_date': end_date,
                        'day': day_name,
                        'event_title': title,
                        'source_page': page_num
                    })
    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError('No calendar events extracted. Please check that the uploaded PDF is the CAIAS calendar.')
    df['department'] = df['department'].str.replace(r'\s+', ' ', regex=True).str.strip()
    df['event_title'] = df['event_title'].str.replace(r'\s+', ' ', regex=True).str.strip()
    df = df.drop_duplicates(subset=['department','start_date','end_date','event_title']).reset_index(drop=True)
    df['search_text'] = (df['department'].fillna('') + ' ' + df['event_title'].fillna('')).str.lower()
    return df

events = extract_events(PDF_PATH)
print(f'Extracted {len(events)} calendar rows. Please inspect the preview below.')
display(events[['department','start_date','end_date','event_title','source_page']].head(30))


Extracted 480 calendar rows. Please inspect the preview below.


,department,start_date,end_date,event_title,source_page
0,IQAC,2026-07-01,2026-07-01,General Staff Meeting(GSM),2
1,IQAC,2026-07-02,2026-07-08,FDP: Vision to Value: Faculty as Partners in I...,2
2,Management,2026-07-03,2026-07-03,Guest Lecture,2
3,Holiday,2026-07-05,2026-07-05,Unspecified event,2
4,Management,2026-07-09,2026-07-09,"Nexus Club Activity: Biz Quest: Think, Compete...",2
5,NCC,2026-07-11,2026-07-11,Inter-Battalion Best Cadet Competition,2
6,Holiday,2026-07-11,2026-07-11,2nd Saturday,2
7,Holiday,2026-07-12,2026-07-12,Unspecified event,2
8,IQAC,2026-07-14,2026-07-14,"Faculty Enrichment Session on ""Understanding S...",2
9,Management,2026-07-14,2026-07-14,Industry Interaction on Integrating AI in Digi...,2


## 3. Optional: import event details from email
For the deadline-friendly prototype, you can paste email text or upload a CSV with columns such as `subject`, `date`, `body`. This lets the demo show calendar/email matching without claiming to have live access to every student's mailbox.

For live Gmail API access, the account owner must grant permission. Do not upload passwords, OAuth client secrets to a public repository, or private emails that you do not have permission to use.

In [ ]:
import io
email_records = []
email_df = pd.DataFrame(columns=['subject','date','body'])

def load_email_csv(file_obj):
    global email_df, email_records
    if file_obj is None:
        return 'No CSV uploaded. You can still use calendar-only recommendations.'
    email_df = pd.read_csv(file_obj.name if hasattr(file_obj, 'name') else file_obj)
    for col in ['subject','date','body']:
        if col not in email_df.columns: email_df[col] = ''
    email_df = email_df[['subject','date','body']].fillna('')
    email_records = email_df.to_dict('records')
    return f'Loaded {len(email_df)} email records.'

def add_pasted_email(subject, body, email_date=''):
    global email_df, email_records
    new_row = {'subject': subject or '', 'date': email_date or '', 'body': body or ''}
    email_records.append(new_row)
    email_df = pd.DataFrame(email_records, columns=['subject','date','body'])
    return f'Added email text. Total email records: {len(email_records)}.'

In [ ]:
from google.colab import files
print('Optional: upload a CSV containing subject, date, body columns. Cancel/skip if you do not have email data.')
try:
    uploaded_emails = files.upload()
    csv_names = [n for n in uploaded_emails if n.lower().endswith('.csv')]
    if csv_names:
        email_df = pd.read_csv(csv_names[0]).fillna('')
        for col in ['subject','date','body']:
            if col not in email_df.columns: email_df[col] = ''
        email_df = email_df[['subject','date','body']]
        email_records = email_df.to_dict('records')
        print(f'Loaded {len(email_records)} email records.')
    else:
        print('No CSV selected. Calendar-only mode is ready.')
except Exception as e:
    print('Email import skipped:', e)


Optional: upload a CSV containing subject, date, body columns. Cancel/skip if you do not have email data.


No CSV selected. Calendar-only mode is ready.


## 4. Recommendation engine
This uses TF-IDF and cosine similarity to compare the student's chosen interests with event names and departments. Keyword matching adds a small boost when an event clearly matches an interest.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

INTEREST_OPTIONS = [
    'Artificial Intelligence','Coding and Programming','Web Development','Cybersecurity',
    'Robotics and IoT','Data Science','Hackathons and Ideathons','Business and Entrepreneurship',
    'Finance and Stock Market','Career and Placements','Communication Skills','Science',
    'Sports and Fitness','Cultural Events','Volunteering and Social Outreach','Research',
    'Psychology and Well-being','Literature and Writing','Environmental Sustainability','General Events'
]

INTEREST_TERMS = {
 'Artificial Intelligence':['ai','artificial intelligence','machine learning','data','digital'],
 'Coding and Programming':['coding','programming','software','computer','it club','technovanza'],
 'Web Development':['web','website','development','software','technology'],
 'Cybersecurity':['cyber','security','cyber ethics','cyber safe','privacy'],
 'Robotics and IoT':['robot','robotic','iot','internet of things','electronics'],
 'Data Science':['data','analytics','power bi','tableau','visualization','research'],
 'Hackathons and Ideathons':['hackathon','ideathon','idea challenge','innovation','bootcamp','design thinking'],
 'Business and Entrepreneurship':['business','entrepreneur','startup','trade fair','management','biz quest'],
 'Finance and Stock Market':['finance','financial','stock market','wealth','investment','tax','commerce'],
 'Career and Placements':['career','placement','employability','interview','resume','linkedin','aptitude'],
 'Communication Skills':['communication','writing','email etiquette','leadership','soft skills'],
 'Science':['science','physics','mathematics','scientia','research','laboratory'],
 'Sports and Fitness':['sports','football','karate','marathon','fitness','tournament','wellness'],
 'Cultural Events':['cultural','dance','fest','festival','music','rangotsav','sangamotsav','celebration'],
 'Volunteering and Social Outreach':['outreach','nss','ncc','volunteer','community','plantation','social','ngo'],
 'Research':['research','conference','seminar','literature review','publication','patent'],
 'Psychology and Well-being':['psychology','well-being','wellness','mental health','kindness','counselling'],
 'Literature and Writing':['literature','writing','storytelling','hindi','kannada','books','journalism'],
 'Environmental Sustainability':['sustainable','sustainability','environment','plantation','zero hunger','green'],
 'General Events':[]
}

def make_event_corpus(df):
    return (df['department'].fillna('') + ' ' + df['event_title'].fillna('')).tolist()

def recommend_events(student_email, interests, top_n=12, include_past=False):
    if not student_email or not student_email.lower().endswith('@caias.in'):
        raise ValueError('Please enter a college email ending in @caias.in. This demo check is not secure authentication.')
    if not interests:
        interests = ['General Events']
    df = events.copy()
    today = pd.Timestamp.now().normalize()
    if not include_past:
        df = df[df['end_date'] >= today].copy()
    if df.empty:
        return pd.DataFrame(columns=['Event','Department','Start date','End date','Match score','Why recommended','Email information'])
    event_texts = (df['department'].fillna('') + ' ' + df['event_title'].fillna('')).tolist()
    profile = ' '.join(interests) + ' ' + ' '.join(t for interest in interests for t in INTEREST_TERMS.get(interest, []))
    corpus = event_texts + [profile]
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
    matrix = vectorizer.fit_transform(corpus)
    scores = cosine_similarity(matrix[:-1], matrix[-1]).ravel()
    lower_texts = [t.lower() for t in event_texts]
    for i, txt in enumerate(lower_texts):
        hits = sum(1 for interest in interests for term in INTEREST_TERMS.get(interest, []) if term in txt)
        scores[i] += min(hits * 0.035, 0.18)
    df['score'] = scores
    df = df.sort_values(['score','start_date'], ascending=[False,True]).head(int(top_n)).copy()
    def why(row):
        txt = (str(row['department']) + ' ' + str(row['event_title'])).lower()
        matched = []
        for interest in interests:
            terms = INTEREST_TERMS.get(interest, [])
            if any(term in txt for term in terms) or interest.lower() in txt:
                matched.append(interest)
        if matched: return 'Matches: ' + ', '.join(matched[:3])
        return 'Recommended from similarity to your selected interests'
    def email_info(row):
        found = []
        event_words = set(re.findall(r'[a-z]{4,}', str(row['event_title']).lower()))
        for em in email_records:
            blob = (str(em.get('subject','')) + ' ' + str(em.get('body',''))).lower()
            if row['event_title'].lower() in blob or (event_words and len(event_words.intersection(set(re.findall(r'[a-z]{4,}', blob)))) >= max(2, min(3, len(event_words)))):
                excerpt = (str(em.get('subject','')) + ' — ' + str(em.get('body',''))).strip(' —')
                if excerpt: found.append(excerpt[:240])
        return ' | '.join(found[:2]) if found else 'No matching email imported'
    df['Why recommended'] = df.apply(why, axis=1)
    df['Email information'] = df.apply(email_info, axis=1)
    df['Match score'] = (df['score'].clip(0,1)*100).round().astype(int).astype(str) + '% (relative)'
    df['Start date'] = df['start_date'].dt.strftime('%d %b %Y')
    df['End date'] = df['end_date'].dt.strftime('%d %b %Y')
    return df.rename(columns={'event_title':'Event','department':'Department'})[['Event','Department','Start date','End date','Match score','Why recommended','Email information','source_page']].rename(columns={'source_page':'Calendar PDF page'})

## 5. Check extracted calendar data
Use this preview before presenting. If the parser has combined or split any event title incorrectly, correct the affected row in `events` below before launching the website.

In [ ]:
display(events[['department','start_date','end_date','event_title','source_page']].sample(min(20, len(events)), random_state=7))
print('Rows:', len(events), '| Departments:', events['department'].nunique())
print('Note: this is best-effort PDF extraction; visually verify rows before relying on them.')

,department,start_date,end_date,event_title,source_page
118,Centre for Counselling & Health Services,2026-09-10,2026-09-10,Suicide Prevention Week,10
448,Commerce,2027-04-17,2027-04-17,Alumni Talk,34
452,Science,2027-04-20,2027-04-20,Career Guidance Programme,34
378,Staff Welfare Association,2027-03-03,2027-03-03,Manager's Day,30
300,Management,2027-01-11,2027-01-14,Competitive Exam Coaching,24
462,Arts & Humanities,2027-05-03,2027-05-03,Interaction -End Semester,35
120,Sports Committee,2026-09-10,2026-09-11,St. Canisius Under-17 Inter-School Sports Fest,10
141,Commerce,2026-09-19,2026-09-19,Commerce Based Documentary,12
429,IQAC,2027-04-03,2027-04-03,General Staff Meeting(GSM),33
13,Management,2026-07-17,2026-07-17,Alumni Talk,3


Rows: 480 | Departments: 32
Note: this is best-effort PDF extraction; visually verify rows before relying on them.


## 6. Launch the student website
The demo login checks the email suffix and is only for showcasing the interface. Do not describe it as verified Google authentication.

In [ ]:
import gradio as gr

def student_dashboard(email, interests, top_n, show_past):
    try:
        recs = recommend_events(email, interests, top_n=top_n, include_past=show_past)
        if recs.empty:
            return 'No matching events found for the selected date range.', recs
        intro = f'Welcome, {email}! Found {len(recs)} recommended events. Scores are relative ranking scores, not probabilities.'
        return intro, recs
    except Exception as e:
        return f'Error: {e}', pd.DataFrame()

with gr.Blocks(title='AI Event Assistant for CAIAS', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎓 AI Event Assistant for CAIAS')
    gr.Markdown('Discover college events selected for your interests. Calendar source: CAIAS Academic Calendar 2026–2027.')
    with gr.Row():
        email_in = gr.Textbox(label='College email (demo check only)', placeholder='25bba890@caias.in')
        interests_in = gr.Dropdown(choices=INTEREST_OPTIONS, multiselect=True, label='Choose your interests', value=['Coding and Programming','Artificial Intelligence'])
    with gr.Row():
        top_n_in = gr.Slider(minimum=5, maximum=30, value=12, step=1, label='Number of recommendations')
        show_past_in = gr.Checkbox(label='Include past events', value=False)
    run_btn = gr.Button('Get My Personalised Events', variant='primary')
    status_out = gr.Markdown()
    results_out = gr.Dataframe(label='Recommended events', interactive=False, wrap=True)
    run_btn.click(student_dashboard, inputs=[email_in, interests_in, top_n_in, show_past_in], outputs=[status_out, results_out])
    gr.Markdown('### Email matching\nEmail information appears only when an authorised email export or sample email text has been imported into this notebook. This demo does not read the visitor’s mailbox automatically.')

demo.launch(share=True, debug=True)

/tmp/ipykernel_1958/1011854953.py:13: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title='AI Event Assistant for CAIAS', theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://acbc79576b6981885d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. Optional Gmail API setup notes
To connect a Gmail mailbox from Colab for a controlled demonstration:
1. In Google Cloud Console, create/select a project and enable the Gmail API.
2. Configure the OAuth consent screen and create an OAuth Client ID for a Desktop app (for a notebook-based test).
3. Download the OAuth client JSON and upload it privately into your Colab session as `credentials.json`.
4. Use Gmail API read-only scope and only access mail for the account owner who explicitly authorises the app.
5. Do not commit `credentials.json` or `token.json` to GitHub or include them in your submission.

**For real student-by-student Gmail access:** this Colab notebook + temporary Gradio link is not a production deployment. Register a web OAuth client, configure the exact redirect URI on a stable host, verify the app/domain as required by Google, and have the Workspace administrator allow access if needed. Request only the minimum scopes necessary.